# Coastal flood step 12: increased-damage hotspot maps (min vs max)

Shows only locations where mangroves are associated with **increased expected annual damage** (`Avoided_EAD_USD < 0`), for:
- minimum scenario
- maximum scenario

Uses cached map layers from step 11 for speed.


In [ ]:
from pathlib import Path
import numpy
import pandas
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LogNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# Paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
cache_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison/cache'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

cache_files = {
    'minimum': cache_dir / 'minimum_all_sector_map_layer.geoparquet',
    'maximum': cache_dir / 'maximum_all_sector_map_layer.geoparquet',
}

for scenario_name, f in cache_files.items():
    if not f.exists():
        raise FileNotFoundError(
            f"Missing cache file for {scenario_name}: {f}\n"
            "Run coastal_flood_11 notebook once to create caches."
        )

if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

print('Using cache files:')
for k, f in cache_files.items():
    print('-', k, '->', f)


In [ ]:
# Load cached layers and keep only increased-damage features
scenario_gdfs = {}
for scenario_name, f in cache_files.items():
    gdf = geopandas.read_parquet(f)
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:3448')

    gdf['Avoided_EAD_USD'] = pandas.to_numeric(gdf['Avoided_EAD_USD'], errors='coerce').fillna(0.0)
    inc = gdf[gdf['Avoided_EAD_USD'] < 0].copy()
    inc['Increased_Damage_USD'] = -inc['Avoided_EAD_USD']
    scenario_gdfs[scenario_name] = inc

    print(f"{scenario_name}: increased-damage features = {len(inc):,}")
    if len(inc) > 0:
        print(
            f"{scenario_name}: increased-damage range USD = "
            f"{inc['Increased_Damage_USD'].min():,.4f} to {inc['Increased_Damage_USD'].max():,.2f}"
        )

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')


In [ ]:
# Shared scale for comparability between minimum and maximum maps
manual_shared_cap_usd = None   # e.g., set 800.0 for fixed range
display_quantile = 0.995

# Visibility controls
use_log_scale = False
manual_log_floor_usd = None    # e.g., 1.0; if None uses pooled median
hotspot_quantile = 0.95        # overlays clear hotspot outlines

pooled_inc = pandas.concat([
    scenario_gdfs['minimum']['Increased_Damage_USD'],
    scenario_gdfs['maximum']['Increased_Damage_USD'],
], ignore_index=True)

if len(pooled_inc) == 0:
    raise ValueError('No increased-damage features found in either scenario.')

quantile_cap_usd = float(pooled_inc.quantile(display_quantile))
if manual_shared_cap_usd is None:
    shared_cap_usd = quantile_cap_usd
else:
    shared_cap_usd = float(abs(manual_shared_cap_usd))

if shared_cap_usd <= 0:
    shared_cap_usd = float(pooled_inc.max()) if float(pooled_inc.max()) > 0 else 1.0

if shared_cap_usd >= 1e6:
    unit_factor = 1e6
    unit_label = 'USD millions'
elif shared_cap_usd >= 1e3:
    unit_factor = 1e3
    unit_label = 'USD thousands'
else:
    unit_factor = 1.0
    unit_label = 'USD'

shared_cap = shared_cap_usd / unit_factor

if use_log_scale:
    auto_floor_usd = float(pooled_inc.quantile(0.50))
    if manual_log_floor_usd is None:
        log_floor_usd = max(1.0, auto_floor_usd)
    else:
        log_floor_usd = float(abs(manual_log_floor_usd))

    if log_floor_usd >= shared_cap_usd:
        log_floor_usd = max(shared_cap_usd / 50.0, shared_cap_usd * 0.1)

    shared_floor = log_floor_usd / unit_factor
    norm = LogNorm(vmin=shared_floor, vmax=shared_cap)
else:
    log_floor_usd = None
    shared_floor = 0.0
    norm = Normalize(vmin=0.0, vmax=shared_cap)

hotspot_threshold_usd = float(pooled_inc.quantile(hotspot_quantile))
hotspot_threshold_usd = min(hotspot_threshold_usd, shared_cap_usd)

print(f'Pooled increased-damage cap q={display_quantile:.3f}: {quantile_cap_usd:,.2f} USD')
print(f'Shared display cap used: {shared_cap_usd:,.2f} USD ({shared_cap:,.2f} {unit_label})')
print(f'Hotspot threshold q={hotspot_quantile:.2f}: {hotspot_threshold_usd:,.2f} USD')
if use_log_scale:
    print(f'Color scale mode: LOG, floor={log_floor_usd:,.2f} USD ({shared_floor:,.2f} {unit_label})')
else:
    print('Color scale mode: LINEAR')


In [ ]:
# Color map: stronger red contrast for increased damage intensity
cmap = LinearSegmentedColormap.from_list(
    'pink_to_darkred',
    ['#fff5f0', '#fcbba1', '#fb6a4a', '#cb181d', '#67000d'],
    N=256,
)


def _plot_geom_by_type(ax, gdf, color=None, linewidth_poly=1.2, linewidth_line=2.0, point_size=34, z=5, alpha=0.96):
    if gdf.empty:
        return
    gt = gdf.geometry.geom_type.astype(str)
    polys = gdf[gt.str.contains('Polygon', na=False)]
    lines = gdf[gt.str.contains('LineString', na=False)]
    points = gdf[gt.str.contains('Point', na=False)]
    if not polys.empty:
        polys.boundary.plot(ax=ax, color=color, linewidth=linewidth_poly, alpha=alpha, zorder=z)
    if not lines.empty:
        lines.plot(ax=ax, color=color, linewidth=linewidth_line, alpha=alpha, zorder=z)
    if not points.empty:
        points.plot(ax=ax, color=color, markersize=point_size, alpha=alpha, zorder=z)


def draw_increase_panel(ax, gdf, scenario_name):
    panel = gdf.copy()
    panel['_plot_val_raw'] = panel['Increased_Damage_USD'] / unit_factor
    if use_log_scale:
        panel['_plot_val'] = panel['_plot_val_raw'].clip(shared_floor, shared_cap)
    else:
        panel['_plot_val'] = panel['_plot_val_raw'].clip(0.0, shared_cap)

    panel['_is_outlier'] = panel['Increased_Damage_USD'] > shared_cap_usd
    panel['_is_hotspot'] = panel['Increased_Damage_USD'] >= hotspot_threshold_usd

    # Base boundary
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    # Plot by geometry type
    geom_type = panel.geometry.geom_type.astype(str)
    polys = panel[geom_type.str.contains('Polygon', na=False)]
    lines = panel[geom_type.str.contains('LineString', na=False)]
    points = panel[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.10, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.95, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

    # Hotspot overlay (q95+) for easier visibility
    hotspots = panel[panel['_is_hotspot']].copy()
    _plot_geom_by_type(ax, hotspots, color='#b22222', linewidth_poly=0.9, linewidth_line=1.3, point_size=24, z=5, alpha=0.90)

    # Stronger outlier overlay
    outliers = panel[panel['_is_outlier']].copy()
    _plot_geom_by_type(ax, outliers, color='#5c0b0b', linewidth_poly=1.2, linewidth_line=2.0, point_size=36, z=6, alpha=0.97)

    # Label top outliers by asset to reduce clutter
    if not outliers.empty:
        key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
        top = outliers.sort_values('Increased_Damage_USD', ascending=False).drop_duplicates(subset=key_cols).head(10)
        label_pts = top.geometry.representative_point()
        for (_, row), pt in zip(top.iterrows(), label_pts):
            v_scaled = row['Increased_Damage_USD'] / unit_factor
            ax.text(
                pt.x,
                pt.y,
                f"{v_scaled:,.1f}",
                fontsize=7,
                color='#5c0b0b',
                ha='left',
                va='bottom',
                zorder=7,
                bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': '#5c0b0b', 'pad': 0.35}
            )

    # Tight map extent
    minx, miny, maxx, maxy = jamaica_boundary.total_bounds
    pad_x = (maxx - minx) * 0.003
    pad_y = (maxy - miny) * 0.001
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    n_hot = int(panel['_is_hotspot'].sum())
    n_out = int(panel['_is_outlier'].sum())
    ax.set_title(f"{scenario_name.capitalize()} | hotspots (q95+): {n_hot:,} | outliers: {n_out:,}", fontsize=12)
    ax.set_axis_off()

    return outliers


In [ ]:
# Build final comparison figure: min and max increased-damage maps
fig, axes = plt.subplots(2, 1, figsize=(11, 10.4), sharex=True, sharey=True)

outliers_min = draw_increase_panel(axes[0], scenario_gdfs['minimum'], 'minimum')
outliers_max = draw_increase_panel(axes[1], scenario_gdfs['maximum'], 'maximum')

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cax = fig.add_axes([0.16, 0.038, 0.68, 0.018])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
tick_vals = numpy.linspace(0.0, shared_cap, 5)
cbar.set_ticks(tick_vals)
cbar.set_ticklabels([f"{t:,.0f}" if shared_cap >= 10 else f"{t:,.2f}" for t in tick_vals])

cbar.set_label(
    f'Increased damage EAD ({unit_label}) | fixed shared range 0 to {shared_cap:,.2f} | darker red = larger increase'
)

fig.suptitle('Locations of increased expected annual damage across all sectors', fontsize=15, y=0.975)
fig.subplots_adjust(top=0.968, bottom=0.065, hspace=0.0)

out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison'
out_dir.mkdir(parents=True, exist_ok=True)

out_png = out_dir / 'increased_damage_locations_all_sectors_min_vs_max.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print('Saved map:', out_png)
plt.show()


In [ ]:
# Save outlier tables and quick summary
out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison'

for scenario_name, outliers in [('minimum', outliers_min), ('maximum', outliers_max)]:
    out_csv = out_dir / f'increased_damage_outliers_all_sectors_{scenario_name}_shared_q{int(display_quantile*1000)}.csv'
    if outliers is None or outliers.empty:
        pandas.DataFrame(columns=['Sector','Subsector','Asset','Layer','Asset_ID','Increased_Damage_USD']).to_csv(out_csv, index=False)
    else:
        key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
        tbl = outliers.sort_values('Increased_Damage_USD', ascending=False).drop_duplicates(subset=key_cols)
        rp = tbl.geometry.representative_point()
        tbl['label_x'] = rp.x
        tbl['label_y'] = rp.y
        keep_cols = [
            'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
            'Increased_Damage_USD', 'Avoided_EAD_USD',
            'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD',
            'label_x', 'label_y'
        ]
        tbl[keep_cols].to_csv(out_csv, index=False)
    print('Saved outliers:', out_csv)

summary = []
for scenario_name, gdf in scenario_gdfs.items():
    v = gdf['Increased_Damage_USD'].fillna(0.0)
    n = len(v)
    n_out = int((v > shared_cap_usd).sum())
    summary.append({
        'Scenario': scenario_name,
        'Increased_Damage_Features': n,
        'Outliers_Count': n_out,
        'Outliers_Percent': 100.0 * n_out / n if n else numpy.nan,
        'Max_Increased_Damage_USD': float(v.max()) if n else numpy.nan,
    })

summary_df = pandas.DataFrame(summary)
display(summary_df)
print(f'Shared cap used (USD): {shared_cap_usd:,.2f}')
